# Conditions géographiques de la nappe

In [ ]:
%%capture
# Étapes précédentes : exécutées sans afficher leurs figures ni leurs messages, et une seule fois par noyau
# (si les données ne sont pas encore en cache, exécuter d'abord donnees.ipynb seul pour suivre le téléchargement)
if "ETAPE_01" not in globals():
    %run ./donnees.ipynb
plt.close("all")

Les nappes phréatiques sont affectées majoritairement par 3 choses :  
- L'altitude, la topographie et la géologie  
- La pluie  
- L'industrie

## I. On va d'abord s'intéresser aux conditions géographiques de la nappe

## 1. Prise en compte de l'effet de l'altitude de la nappe

1. Commençons par un état des lieux de l'altitude des différentes stations.

In [ ]:
stations['altitude_station'] = pd.to_numeric(stations['altitude_station'])
points_carte = stations.to_crs(epsg=3857)   #on transforme le geopandas avec le code EPSG utilisé par les cartes et Google

fig, ax = plt.subplots(figsize=(12, 10))
points_carte.plot(ax=ax,
    column='altitude_station', # variable qui donne la couleur du point
    cmap='terrain', # fond de carte pour la topographie
    legend=True,                 # Affichage de la barre de gradient
    legend_kwds={
        'label': "Altitude de la station (en mètres)",
        'shrink': 0.7   #Limite la taille de la barre de la légende
    },
    zorder=2
)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldTopoMap, zorder=1)   #Fonds de carte
plt.title("Altitude des stations hydrologiques")

2. Annulation de l'effet de l'altitude sur le niveau des nappes.

La variable niveau_nappe_eau nous donne le niveau de l'eau de la nappe par rapport au niveau de la mer. Pas par rapport au fond de la nappe. On ne peut pas obtenir le volume d'eau dans la nappe. On ne peut qu'observer le niveau de l'eau dans la nappe.  
Nous allons donc étudier les variations du niveau de la nappe sur la moyenne historique afin de déterminer si l'industrie creuse les nappes.  
On va annuler l'effet de l'altitude sur le niveau de l'eau dans la station pour pouvoir comparer les niveau d'eau dans les différentes stations.  
Pour cela, on va centrer le niveau pour ne garder que les variations de niveau des nappes :  
Exemple : si on a en moyenne une hauteur de 100m pour une station, après une sécheresse qui fait baisser ce niveau à 90m, on aura -10 de variation sur notre chronique.

In [ ]:
moyenne_par_station = chroniques.groupby("code_bss")["niveau_nappe_eau"].mean().reset_index()
moyenne_par_station = moyenne_par_station.rename(columns={"niveau_nappe_eau": "moyenne_de_station"})
chroniques = chroniques.merge(moyenne_par_station, on="code_bss", how="left")   #On rajoute la colonne dans le DataFrame de base en affectant à chaque ligne d'observation une colonne avec la moyenne de sa station

chroniques["niveau_eau_isole"] = chroniques["niveau_nappe_eau"] - chroniques["moyenne_de_station"]

print(chroniques[["code_bss", "date_mesure", "niveau_nappe_eau", "niveau_eau_isole"]].sample(15))

## 2. Prise en compte de l'effet de la nature des sols

On va vérifier si l'effet de la géologie doit être pris en compte.  
En effet, on peut supposer que selon la nature du sol, les nappes phréatiques peuvent se vider plus ou moins rapidement et les variations de niveau peuvent donc être impactés.  
Il y a de nombreux types de sols. On va donc se concentrer sur les 5 les plus courants pour vérifier notre hypothèse  

In [ ]:
#Nettoyage de type :
stations['noms_masse_eau_edl'] = stations['noms_masse_eau_edl'].astype(str)

#Calcul des variations du niveau de chaque nappe
amplitude = chroniques.groupby('code_bss')['niveau_nappe_eau'].agg(lambda x: x.max() - x.min()).reset_index()
amplitude = amplitude.rename(columns={'niveau_nappe_eau': 'amplitude_metres'})
stations = stations.merge(amplitude, on='code_bss', how='inner')

#On prend les 5 sols les plus fréquents
compte = stations.groupby('noms_masse_eau_edl').size()
tri = compte.sort_values(ascending=False)
top_5_sols = tri.head(5).index

stations_top = stations[stations['noms_masse_eau_edl'].isin(top_5_sols)]
stations_top.boxplot(column='amplitude_metres', by='noms_masse_eau_edl', grid=False, rot=15, figsize=(12, 6))
plt.title("Sensibilité aux variations de la nappe selon la géologie.")
plt.suptitle("") # sinon pPandas met un sous-titre automatique
plt.ylabel("Variation (max - min en mètres)")
plt.xlabel("Nature géologique de la nappe")
plt.show()

**Lecture.** Les médianes vont d'environ 1 m (craie du Cambrésis) à 6 m (craie des vallées de la Scarpe et de la Sensée) : les trois craies couvrent à elles seules presque tout l'éventail, de l'amplitude la plus faible à la plus forte. La géologie ne donne donc pas un ordre simple des amplitudes. Attention : l'amplitude est calculée sur la période mesurée de chaque station, qui n'est pas la même pour toutes.

On va créer une colonne qui met des labels sur les grandes familles géologiques. On pourra ainsi prendre en compte cet effet dans la suite de notre étude.

In [ ]:
#Familles de sols
conditions_geo = [
    stations['noms_masse_eau_edl'].str.contains('Craie', case=False, na=False),
    stations['noms_masse_eau_edl'].str.contains('Sable|Landénien', case=False, na=False),
    stations['noms_masse_eau_edl'].str.contains('Calcaire', case=False, na=False)
]
labels_type_station = ['Craie', 'Sable', 'Calcaire']

#On ajoute la colonne au dataframe
stations['famille_geologique'] = np.select(conditions_geo, labels_type_station, default='Autre')
print(stations['famille_geologique'].value_counts())

Après vérification, dans la catégorie autre, on ne retrouve quasiment que des types de sols NaN.

## 3. Prise en compte de l'effet de la topologie (distance entre la surface et la nappe)

On va vérifier si l'effet de la topologie a un effet (à annuler pour notre analyse) sur le niveau des nappes phréatiques

In [ ]:
plt.figure(figsize=(10, 6))
# On limite l'axe X à 150m pour ne pas écraser le graphique avec les extrêmes
donnees_visuelles = stations[stations['profondeur_investigation'] < 150]

plt.scatter(donnees_visuelles['profondeur_investigation'], donnees_visuelles['amplitude_metres'])
plt.title("Relation entre profondeur de la nappe et violence des variations")
plt.xlabel("Profondeur de la station (en mètres)")
plt.ylabel("Amplitude de variation (en mètres)")
plt.grid(True)
plt.show()

**Lecture.** Il n'y a pas de relation nette entre la profondeur de l'ouvrage et l'amplitude. Les piézomètres de moins de 10 m varient peu (1 à 4 m). Les plus fortes amplitudes se trouvent entre 20 et 50 m de profondeur (jusqu'à 11 m), avec un point isolé à 17,6 m vers 90 m. Au-delà de 80 m, les amplitudes restent modérées (1 à 4 m). La profondeur seule ne prédit donc pas l'ampleur des variations. (Il s'agit de la profondeur de l'ouvrage, pas de celle de l'eau.)

In [ ]:
#Nouvelle colonne qui donne un label topologie à la station
stations['classe_profondeur'] = 'Non renseignée'

stations.loc[stations['profondeur_investigation'] < 30, 'classe_profondeur'] = 'Superficielle (<30m)'
stations.loc[(stations['profondeur_investigation'] >= 30) & (stations['profondeur_investigation'] <= 80), 'classe_profondeur'] = 'Intermédiaire (30-80m)'
stations.loc[stations['profondeur_investigation'] > 80, 'classe_profondeur'] = 'Profonde (>80m)'

print(stations['classe_profondeur'].value_counts())

On voit qu'on a 40 stations superficielles. Notre hypothèse de départ est qu'elles devraient être les plus affectées par les installations industrielles, et que les stations profondes, moins exposées, pourraient servir de groupe témoin.  
La partie III montrera que c'est l'inverse : une fois la pluie retirée, ce sont les nappes profondes qui baissent le plus.

## Recentrage sur les stations intéressantes
On remarque qu'on n'a plus que environ 70 stations alors qu'on en avait 730 au début.  
Cela est dû au inner join fait pour catégoriser les stations selon la géologie (avec le calcul de l'amplitude pour chaque type de sol).  
Vérifions à quoi ressemble les stations éliminées et leurs chroniques pour comprendre.

In [ ]:
#On reprend la liste de toutes les stations
stations_brutes = pd.read_csv(chemin_stations)

codes_valides = chroniques['code_bss'].unique() #Stations qui ont été conservées dans le processus.
print("Stations conservées dans l'analyse : ", len(codes_valides))
print(stations[['code_bss', 'nom_commune', 'nb_mesures_piezo']].sort_values(by="nb_mesures_piezo", ascending=False).head(10))
stations_eliminees = stations_brutes[~stations_brutes['code_bss'].isin(codes_valides)]

#On regarde le nombre de mesures pour les stations éliminées :
print("Stations éliminées : ", len(stations_eliminees))
print(stations_eliminees[['code_bss', 'nom_commune', 'nb_mesures_piezo']].sort_values(by="nb_mesures_piezo", ascending=False).head(10))

Donc on a un problème manifestement.  
Il est possible que les stations éliminées sont des stations qui n'ont pas enregistré de données entre 2015 et 2026 (car le nb_mesures_piezo est le nombre de mesures totales depuis la création de la station).

In [ ]:
stations_eliminees['date_fin_mesure'] = pd.to_datetime(stations_eliminees['date_fin_mesure'])
#on regarde l'année d'arrêt de la station
stations_eliminees['annee_arret'] = stations_eliminees['date_fin_mesure'].dt.year

# On affiche l'année de fin d'activité pour les 10 plus grosses stations éliminées
colonnes_visibles = ['code_bss', 'nom_commune', 'nb_mesures_piezo', 'annee_arret']
print(stations_eliminees.sort_values(by="nb_mesures_piezo", ascending=False)[colonnes_visibles].head(10))

arret_avant_2015 = len(stations_eliminees[stations_eliminees['annee_arret'] < 2015])
jamais_mesure = len(stations_eliminees[stations_eliminees['annee_arret'].isna()])
print("Stations arrêtées définitivement avant 2015 : ", arret_avant_2015)
print("Stations sans date de fin : ", jamais_mesure)

print(stations_eliminees.loc[stations_eliminees['annee_arret'].isna()].sort_values(by="nb_mesures_piezo", ascending=False)[colonnes_visibles].head(10))

Donc c'est tout bon : seules les 67 stations gardées sont encore actives après 2015 et rentrent dans le cadre de notre étude.

## Conclusion partielle
Le niveau d'une nappe est une altitude : il dépend d'abord de l'endroit où se trouve la station. Pour comparer les stations entre elles, on étudie donc leurs **variations** (écart à leur moyenne, puis tendance en partie III) plutôt que leur niveau.  
La géologie et la profondeur n'expliquent pas simplement l'ampleur de ces variations : les amplitudes varient autant à l'intérieur d'une même roche qu'entre roches, et sans lien net avec la profondeur.  
On garde ces deux caractéristiques (famille géologique, classe de profondeur) pour les comparer à l'effet de l'industrie en partie III.

In [ ]:
#Marqueur : cette étape est faite, les notebooks qui en dépendent ne la relanceront pas dans ce noyau
ETAPE_03 = True